# Ablation: Neural Network on Raw 3-D Grids vs TurnoverRate

This notebook is an **ablation study** comparing:

| Model | Input | Description |
|---|---|---|
| **3D-CNN** | Raw grids $(15 \times 15 \times 10 \times T)$ | Spatiotemporal convolutions on the neural activity volume |
| **1D-CNN** | TurnoverRate vector $(720\text{-dim})$ | Temporal convolutions on the topological summary |
| **MLP** | TurnoverRate vector $(720\text{-dim})$ | Fully-connected baseline on the topological summary |
| **LogReg** | TurnoverRate vector $(720\text{-dim})$ | Linear baseline (same as notebook 03) |

**Purpose**: Determine whether the topological TurnoverRate representation
captures discriminative information that a powerful neural network applied
directly to the raw data would also find — or whether topology provides a
unique advantage.

All models are evaluated with the same **5-fold stratified CV** per mouse,
using the same splits for fair comparison.

## 1. Setup

In [1]:
import re, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, accuracy_score

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__}, device: {DEVICE}")

PyTorch 2.11.0+cu130, device: cpu


In [2]:
# ── Paths ────────────────────────────────────────────────────────────────────
DATA_ROOT = Path("/orfeo/scratch/area/mbiagetti/project_cc")
GRID_ROOT = Path("/orfeo/scratch/area/ygardinazzi/sensorium_2026/derivatives/grid-15x15x10_norm-by_minmax")
META_ROOT = Path("/u/area/mbiagetti/Codes/sensorium_metadata/metadata")
CACHE_DIR = Path("cache")

P_ACTIVE  = 30
ZZ_FOLDER = f"trials_zz-thresh-{P_ACTIVE}"

mice = sorted(
    d.name for d in DATA_ROOT.iterdir()
    if d.is_dir() and d.name.startswith("dynamic")
)
print(f"Found {len(mice)} mice")

Found 10 mice


In [3]:
# ── Helpers: metadata & turnover cache ────────────────────────────────────────

def load_trial_metadata(mouse_name):
    csv_path = META_ROOT / mouse_name / "trials" / f"meta-trials_{mouse_name}.csv"
    return pd.read_csv(csv_path)


def get_trial_info(mouse_name):
    """Return list of (trial_id, label, valid_frames) for trials with zigzag data."""
    df = load_trial_metadata(mouse_name)
    trial_to_label  = dict(zip(df["trial"].astype(int), df["label"]))
    trial_to_frames = dict(zip(df["trial"].astype(int), df["valid_frames"].astype(int)))

    zz_dir = DATA_ROOT / mouse_name / ZZ_FOLDER
    files = sorted(f for f in zz_dir.glob("zz-thresh-*.npy") if "info" not in f.name)

    trials = []
    for f in files:
        match = re.search(r"trial-(\d+)", f.stem)
        if match is None:
            continue
        tid = int(match.group(1))
        if tid not in trial_to_label:
            continue
        trials.append((tid, trial_to_label[tid], trial_to_frames[tid]))
    return trials


def load_turnover_cache(mouse_name):
    """Load pre-computed TurnoverRate vectors from notebook 03 cache."""
    cache_file = CACHE_DIR / f"turnover_{mouse_name}.npz"
    if not cache_file.exists():
        raise FileNotFoundError(f"Run notebook 03 first to generate {cache_file}")
    data = np.load(cache_file, allow_pickle=True)
    return data["X"], data["labels"], int(data["clip_frames"])

In [4]:
# ── Dataset classes ───────────────────────────────────────────────────────────

class GridDataset(Dataset):
    """Lazily loads 4-D grid volumes (15x15x10xT) from disk per trial."""

    def __init__(self, mouse_name, trial_ids, labels, clip_frames):
        self.mouse = mouse_name
        self.trial_ids = trial_ids
        self.clip_frames = clip_frames
        self.le = LabelEncoder().fit(sorted(set(labels)))
        self.labels = self.le.transform(labels)
        self.n_classes = len(self.le.classes_)

        # Build file path template
        grid_dir = GRID_ROOT / mouse_name / "trials"
        # Find the actual filename pattern
        sample = next(grid_dir.glob("grid-*.npy"))
        # Extract prefix pattern
        self.template = str(grid_dir / sample.name.split("_trial-")[0]) + "_trial-{}.npy"

    def __len__(self):
        return len(self.trial_ids)

    def __getitem__(self, idx):
        tid = self.trial_ids[idx]
        path = self.template.format(tid)
        grid = np.load(path)  # (15, 15, 10, T)
        # Clip to uniform length
        grid = grid[:, :, :, :self.clip_frames]  # (15, 15, 10, clip)
        # Rearrange to (C=10, T=clip, H=15, W=15) for Conv3d (or treat as channels)
        # We'll use (1, 15, 15, 10, T) -> batch of 3D volumes over time
        # For 3D-CNN: treat as (C=1, D=15, H=15, W=10*clip) or (C=10, T, 15, 15)
        # Best approach: (1, T, 15, 15, 10) -> 3D conv over (T, H, W) with depth as channels
        # Actually: (10, clip, 15, 15) — depth-slices as channels, temporal+spatial conv
        grid = grid.transpose(2, 3, 0, 1)  # (10, T, 15, 15)
        grid = grid[:, :self.clip_frames, :, :]
        return torch.tensor(grid, dtype=torch.float32), self.labels[idx]


class TurnoverDataset(Dataset):
    """Wraps pre-computed TurnoverRate feature vectors."""

    def __init__(self, X, labels):
        self.le = LabelEncoder().fit(sorted(set(labels)))
        self.X = torch.tensor(X, dtype=torch.float32)
        self.labels = self.le.transform(labels)
        self.n_classes = len(self.le.classes_)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.X[idx], self.labels[idx]

In [5]:
# ── Model definitions ─────────────────────────────────────────────────────────

class CNN3D(nn.Module):
    """3-D CNN operating on raw neural activity grids.

    Input shape: (batch, 10, T, 15, 15)
      - 10 depth slices as input channels
      - T temporal frames
      - 15x15 spatial grid

    Uses 3D convolutions over (T, H, W) then global average pooling.
    """

    def __init__(self, n_classes, clip_frames, dropout=0.3):
        super().__init__()
        self.conv1 = nn.Conv3d(10, 32, kernel_size=(5, 3, 3), padding=(2, 1, 1))
        self.bn1   = nn.BatchNorm3d(32)
        self.conv2 = nn.Conv3d(32, 64, kernel_size=(5, 3, 3), padding=(2, 1, 1))
        self.bn2   = nn.BatchNorm3d(64)
        self.conv3 = nn.Conv3d(64, 128, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.bn3   = nn.BatchNorm3d(128)
        self.pool  = nn.AdaptiveAvgPool3d(1)  # global average pool
        self.drop  = nn.Dropout(dropout)
        self.fc    = nn.Linear(128, n_classes)

    def forward(self, x):
        # x: (B, 10, T, 15, 15)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.max_pool3d(x, kernel_size=(2, 2, 2), ceil_mode=True)
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.max_pool3d(x, kernel_size=(2, 2, 2), ceil_mode=True)
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.pool(x).flatten(1)  # (B, 128)
        x = self.drop(x)
        return self.fc(x)


class CNN1D(nn.Module):
    """1-D CNN on TurnoverRate vectors (temporal convolution per dimension).

    Input: (batch, 720) reshaped to (batch, 3, clip_frames)
    """

    def __init__(self, n_classes, clip_frames, dropout=0.3):
        super().__init__()
        self.clip = clip_frames
        self.n_dim = 3  # H0, H1, H2
        self.conv1 = nn.Conv1d(self.n_dim, 32, kernel_size=7, padding=3)
        self.bn1   = nn.BatchNorm1d(32)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.bn2   = nn.BatchNorm1d(64)
        self.conv3 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.bn3   = nn.BatchNorm1d(64)
        self.pool  = nn.AdaptiveAvgPool1d(1)
        self.drop  = nn.Dropout(dropout)
        self.fc    = nn.Linear(64, n_classes)

    def forward(self, x):
        # x: (B, 720) -> (B, 3, clip)
        B = x.size(0)
        x = x.view(B, self.n_dim, self.clip)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.max_pool1d(x, 2, ceil_mode=True)
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.max_pool1d(x, 2, ceil_mode=True)
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.pool(x).flatten(1)  # (B, 64)
        x = self.drop(x)
        return self.fc(x)


class MLP(nn.Module):
    """Simple MLP on TurnoverRate vectors."""

    def __init__(self, n_classes, input_dim, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
# ── Training & evaluation utilities ───────────────────────────────────────────

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = torch.tensor(y_batch, dtype=torch.long).to(DEVICE)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y_batch)
        correct += (logits.argmax(1) == y_batch).sum().item()
        total += len(y_batch)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(DEVICE)
        logits = model(X_batch)
        all_preds.append(logits.argmax(1).cpu().numpy())
        all_labels.append(np.array(y_batch))
    preds = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro", zero_division=0)
    return acc, f1, preds, labels


def run_nn_cv(make_model_fn, dataset, labels_int, n_splits=5, epochs=40,
              lr=1e-3, batch_size=32, patience=8, verbose=False):
    """Run stratified K-fold CV for a PyTorch model."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    fold_accs, fold_f1s = [], []

    for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels_int)), labels_int)):
        train_loader = DataLoader(Subset(dataset, train_idx),
                                  batch_size=batch_size, shuffle=True,
                                  num_workers=0, pin_memory=False)
        val_loader   = DataLoader(Subset(dataset, val_idx),
                                  batch_size=batch_size, shuffle=False,
                                  num_workers=0, pin_memory=False)

        model = make_model_fn().to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, patience=3, factor=0.5)
        criterion = nn.CrossEntropyLoss()

        best_f1, wait = 0.0, 0
        for epoch in range(epochs):
            train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
            val_acc, val_f1, _, _ = evaluate(model, val_loader)
            scheduler.step(1 - val_f1)

            if val_f1 > best_f1:
                best_f1 = val_f1
                best_acc = val_acc
                wait = 0
            else:
                wait += 1
                if wait >= patience:
                    break

        fold_accs.append(best_acc)
        fold_f1s.append(best_f1)
        if verbose:
            print(f"    Fold {fold+1}: acc={best_acc:.3f}  F1={best_f1:.3f}  "
                  f"(epoch {epoch+1})")

    return {
        "acc": np.mean(fold_accs), "acc_std": np.std(fold_accs),
        "f1":  np.mean(fold_f1s),  "f1_std":  np.std(fold_f1s),
    }

## 2. Load data & build datasets

We load TurnoverRate vectors from the notebook-03 cache, and build a
lazy-loading `GridDataset` for the raw 3-D grids (loaded on-demand to
avoid memory issues: each grid is 15x15x10xT ~ 5MB).

In [7]:
# ── Pick a reference mouse for initial development ───────────────────────────
# Use one of the "complete" mice (720 trials) for the ablation
REF_MOUSE = mice[0]  # dynamic29156... (720 trials, 300 frames)

# Load turnover cache
X_turn, labels_str, clip_frames = load_turnover_cache(REF_MOUSE)
le = LabelEncoder().fit(sorted(set(labels_str)))
labels_int = le.transform(labels_str)
n_classes = len(le.classes_)

print(f"Mouse: {REF_MOUSE[:45]}")
print(f"  Trials: {len(labels_str)}, Clip frames: {clip_frames}")
print(f"  Classes ({n_classes}): {list(le.classes_)}")
print(f"  TurnoverRate shape: {X_turn.shape}")

# Build trial info for grid dataset
trial_info = get_trial_info(REF_MOUSE)
# Filter to same trials that have zigzag data (same order)
trial_ids = [t[0] for t in trial_info]
trial_labels = [t[1] for t in trial_info]

# Verify alignment
assert len(trial_ids) == len(labels_str), \
    f"Mismatch: {len(trial_ids)} grid trials vs {len(labels_str)} turnover trials"
assert all(a == b for a, b in zip(trial_labels, labels_str)), \
    "Label mismatch between grid and turnover data"

# Create datasets
grid_ds = GridDataset(REF_MOUSE, trial_ids, trial_labels, clip_frames)
turn_ds = TurnoverDataset(X_turn, labels_str)

# Quick check
sample_grid, sample_label = grid_ds[0]
sample_turn, _ = turn_ds[0]
print(f"\n  Grid sample shape: {sample_grid.shape}")  # (10, T, 15, 15)
print(f"  Turnover sample shape: {sample_turn.shape}")  # (720,)

Mouse: dynamic29156-11-10-Video-8744edeac3b4d1ce16b6
  Trials: 720, Clip frames: 240
  Classes (4): [np.str_('GaussianDot'), np.str_('NaturalVideo'), np.str_('PinkNoise'), np.str_('RandomDots')]
  TurnoverRate shape: (720, 720)

  Grid sample shape: torch.Size([10, 240, 15, 15])
  Turnover sample shape: torch.Size([720])


## 3. Ablation experiment (single mouse)

Run all four models with the same 5-fold stratified CV splits:

1. **3D-CNN** on raw grids — can the NN learn directly from spatiotemporal activity?
2. **1D-CNN** on TurnoverRate — does temporal conv help beyond the topological summary?
3. **MLP** on TurnoverRate — non-linear baseline on topological features
4. **LogReg** on TurnoverRate — linear baseline (same as notebook 03)

In [8]:
# ── 3a. LogReg baseline (sklearn) ─────────────────────────────────────────────
print("=" * 60)
print("LogReg on TurnoverRate (linear baseline)")
print("=" * 60)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"acc": "accuracy", "f1": "f1_macro"}
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, random_state=42, class_weight="balanced")),
])
cv_res = cross_validate(pipe, X_turn, labels_str, cv=cv, scoring=scoring)
logreg_res = {
    "acc": cv_res["test_acc"].mean(), "acc_std": cv_res["test_acc"].std(),
    "f1":  cv_res["test_f1"].mean(),  "f1_std":  cv_res["test_f1"].std(),
}
print(f"  acc={logreg_res['acc']:.3f} +/- {logreg_res['acc_std']:.3f}")
print(f"  F1 ={logreg_res['f1']:.3f} +/- {logreg_res['f1_std']:.3f}")

LogReg on TurnoverRate (linear baseline)
  acc=0.869 +/- 0.018
  F1 =0.764 +/- 0.023


In [9]:
# ── 3b. MLP on TurnoverRate ───────────────────────────────────────────────────
print("=" * 60)
print("MLP on TurnoverRate")
print("=" * 60)

# Normalise features for NN (same as scaler in LogReg pipeline)
scaler = StandardScaler().fit(X_turn)
X_scaled = scaler.transform(X_turn)
turn_ds_scaled = TurnoverDataset(X_scaled, labels_str)

t0 = time.perf_counter()
mlp_res = run_nn_cv(
    make_model_fn=lambda: MLP(n_classes, input_dim=X_turn.shape[1]),
    dataset=turn_ds_scaled,
    labels_int=labels_int,
    epochs=60, lr=1e-3, batch_size=64, patience=10, verbose=True,
)
elapsed = time.perf_counter() - t0
print(f"\n  MLP: acc={mlp_res['acc']:.3f} +/- {mlp_res['acc_std']:.3f}")
print(f"       F1 ={mlp_res['f1']:.3f} +/- {mlp_res['f1_std']:.3f}")
print(f"       Time: {elapsed:.1f}s")

MLP on TurnoverRate


TypeError: ReduceLROnPlateau.__init__() got an unexpected keyword argument 'verbose'

In [ ]:
# ── 3c. 1D-CNN on TurnoverRate ────────────────────────────────────────────────
print("=" * 60)
print("1D-CNN on TurnoverRate")
print("=" * 60)

t0 = time.perf_counter()
cnn1d_res = run_nn_cv(
    make_model_fn=lambda: CNN1D(n_classes, clip_frames=clip_frames),
    dataset=turn_ds_scaled,
    labels_int=labels_int,
    epochs=60, lr=1e-3, batch_size=64, patience=10, verbose=True,
)
elapsed = time.perf_counter() - t0
print(f"\n  1D-CNN: acc={cnn1d_res['acc']:.3f} +/- {cnn1d_res['acc_std']:.3f}")
print(f"          F1 ={cnn1d_res['f1']:.3f} +/- {cnn1d_res['f1_std']:.3f}")
print(f"          Time: {elapsed:.1f}s")

In [ ]:
# ── 3d. 3D-CNN on raw grids ──────────────────────────────────────────────────
print("=" * 60)
print("3D-CNN on raw neural activity grids")
print("=" * 60)
print(f"  Grid shape per trial: (10, {clip_frames}, 15, 15)")
print(f"  This will be slower — loading grids from disk per batch.")
print()

t0 = time.perf_counter()
cnn3d_res = run_nn_cv(
    make_model_fn=lambda: CNN3D(n_classes, clip_frames=clip_frames),
    dataset=grid_ds,
    labels_int=labels_int,
    epochs=40, lr=5e-4, batch_size=16, patience=8, verbose=True,
)
elapsed = time.perf_counter() - t0
print(f"\n  3D-CNN: acc={cnn3d_res['acc']:.3f} +/- {cnn3d_res['acc_std']:.3f}")
print(f"          F1 ={cnn3d_res['f1']:.3f} +/- {cnn3d_res['f1_std']:.3f}")
print(f"          Time: {elapsed:.1f}s")

## 4. Comparison

In [ ]:
# ── Bar chart comparison ──────────────────────────────────────────────────────
results = {
    "LogReg\n(TurnoverRate)": logreg_res,
    "MLP\n(TurnoverRate)": mlp_res,
    "1D-CNN\n(TurnoverRate)": cnn1d_res,
    "3D-CNN\n(Raw grids)": cnn3d_res,
}

names = list(results.keys())
accs  = [results[n]["acc"]     for n in names]
a_err = [results[n]["acc_std"] for n in names]
f1s   = [results[n]["f1"]      for n in names]
f_err = [results[n]["f1_std"]  for n in names]

x = np.arange(len(names))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w/2, accs, w, yerr=a_err, capsize=4, alpha=0.85, label="Accuracy",
       color=["#4C72B0", "#4C72B0", "#4C72B0", "#DD8452"])
ax.bar(x + w/2, f1s,  w, yerr=f_err, capsize=4, alpha=0.55, label="Macro F1",
       hatch="//", color=["#4C72B0", "#4C72B0", "#4C72B0", "#DD8452"])
ax.axhline(0.25, color="gray", ls="--", alpha=0.5, label="Chance (1/4)")
ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=10)
ax.set_ylabel("Score")
ax.set_title(
    f"Ablation: Neural Network vs Topological TurnoverRate\n"
    f"Mouse {REF_MOUSE.split('-')[0].replace('dynamic','')}, "
    f"5-fold stratified CV, {len(labels_str)} trials",
    fontsize=13,
)
ax.legend(loc="lower right")
ax.set_ylim(0, 1.1)
ax.grid(axis="y", alpha=0.3)

# Annotate bars
for i, (a, f) in enumerate(zip(accs, f1s)):
    ax.text(i - w/2, a + a_err[i] + 0.02, f"{a:.2f}", ha="center", fontsize=8)
    ax.text(i + w/2, f + f_err[i] + 0.02, f"{f:.2f}", ha="center", fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ── Summary table ────────────────────────────────────────────────────────────
print("=" * 72)
print("ABLATION RESULTS — Stimulus Classification")
print(f"Mouse: {REF_MOUSE.split('-')[0].replace('dynamic','')}, "
      f"{len(labels_str)} trials, {n_classes} classes, 5-fold CV")
print("=" * 72)
print(f"{'Model':<25s} | {'Input':<15s} | {'Acc':>12s} | {'F1':>12s}")
print("-" * 72)

rows = [
    ("LogReg",  "TurnoverRate", logreg_res),
    ("MLP",     "TurnoverRate", mlp_res),
    ("1D-CNN",  "TurnoverRate", cnn1d_res),
    ("3D-CNN",  "Raw grids",    cnn3d_res),
]
for name, inp, r in rows:
    print(f"{name:<25s} | {inp:<15s} | "
          f"{r['acc']:.3f}+/-{r['acc_std']:.3f} | "
          f"{r['f1']:.3f}+/-{r['f1_std']:.3f}")

print()
print("Interpretation:")
best_topo = max([logreg_res, mlp_res, cnn1d_res], key=lambda r: r["f1"])
raw_f1 = cnn3d_res["f1"]
topo_f1 = best_topo["f1"]

if topo_f1 > raw_f1 + 0.03:
    print("  -> TurnoverRate features OUTPERFORM the 3D-CNN on raw data.")
    print("     The topological summary captures discriminative structure")
    print("     that the NN cannot easily learn from raw spatiotemporal grids.")
elif abs(topo_f1 - raw_f1) <= 0.03:
    print("  -> TurnoverRate features and 3D-CNN perform COMPARABLY.")
    print("     Both representations capture similar discriminative information,")
    print("     but TurnoverRate achieves it with a 720-dim vector vs 4D volumes.")
else:
    print("  -> 3D-CNN on raw data OUTPERFORMS TurnoverRate features.")
    print("     The raw spatiotemporal data contains additional discriminative")
    print("     information not captured by the topological summary.")

## 5. Multi-mouse ablation

Run the ablation across all mice to check consistency. To keep runtime
manageable, we run only **LogReg** and **3D-CNN** (the extremes) for
all mice, plus **MLP** on TurnoverRate as the NN baseline.

In [ ]:
# ── Multi-mouse ablation ──────────────────────────────────────────────────────
all_results = {}

for mname in mice:
    short = mname.split("-")[0].replace("dynamic", "")
    print(f"\n{'='*60}")
    print(f"Mouse {short}: {mname[:50]}")
    print(f"{'='*60}")

    try:
        X_m, labels_m, clip_m = load_turnover_cache(mname)
    except FileNotFoundError:
        print("  SKIPPED — no turnover cache (run notebook 03 first)")
        continue

    le_m = LabelEncoder().fit(sorted(set(labels_m)))
    labels_m_int = le_m.transform(labels_m)
    nc = len(le_m.classes_)

    # --- LogReg ---
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, random_state=42, class_weight="balanced")),
    ])
    cv_res = cross_validate(pipe, X_m, labels_m, cv=cv, scoring=scoring)
    lr = {"acc": cv_res["test_acc"].mean(), "acc_std": cv_res["test_acc"].std(),
          "f1":  cv_res["test_f1"].mean(),  "f1_std":  cv_res["test_f1"].std()}
    print(f"  LogReg:  acc={lr['acc']:.3f}  F1={lr['f1']:.3f}")

    # --- MLP ---
    sc = StandardScaler().fit(X_m)
    td = TurnoverDataset(sc.transform(X_m), labels_m)
    mlp_r = run_nn_cv(
        lambda nc=nc, dim=X_m.shape[1]: MLP(nc, dim),
        td, labels_m_int, epochs=50, lr=1e-3, batch_size=64, patience=8)
    print(f"  MLP:     acc={mlp_r['acc']:.3f}  F1={mlp_r['f1']:.3f}")

    # --- 3D-CNN ---
    trials_m = get_trial_info(mname)
    tids = [t[0] for t in trials_m]
    tlabels = [t[1] for t in trials_m]

    # Check grid data exists
    grid_dir = GRID_ROOT / mname / "trials"
    if not grid_dir.exists():
        print(f"  3D-CNN: SKIPPED — no grid data at {grid_dir}")
        cnn_r = {"acc": np.nan, "acc_std": np.nan, "f1": np.nan, "f1_std": np.nan}
    elif len(tids) != len(labels_m):
        print(f"  3D-CNN: SKIPPED — trial count mismatch ({len(tids)} vs {len(labels_m)})")
        cnn_r = {"acc": np.nan, "acc_std": np.nan, "f1": np.nan, "f1_std": np.nan}
    else:
        gd = GridDataset(mname, tids, tlabels, clip_m)
        cnn_r = run_nn_cv(
            lambda nc=nc, cf=clip_m: CNN3D(nc, cf),
            gd, labels_m_int, epochs=30, lr=5e-4, batch_size=16, patience=6)
        print(f"  3D-CNN:  acc={cnn_r['acc']:.3f}  F1={cnn_r['f1']:.3f}")

    all_results[mname] = {"LogReg": lr, "MLP": mlp_r, "3D-CNN": cnn_r}

print(f"\n\nCompleted {len(all_results)} mice.")

In [ ]:
# ── Multi-mouse comparison plot ───────────────────────────────────────────────
short_names = []
lr_f1s, mlp_f1s, cnn_f1s = [], [], []

for mname, res in all_results.items():
    short_names.append(mname.split("-")[0].replace("dynamic", ""))
    lr_f1s.append(res["LogReg"]["f1"])
    mlp_f1s.append(res["MLP"]["f1"])
    cnn_f1s.append(res["3D-CNN"]["f1"])

x = np.arange(len(short_names))
w = 0.25

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - w, lr_f1s, w, alpha=0.85, label="LogReg (TurnoverRate)", color="#4C72B0")
ax.bar(x,     mlp_f1s, w, alpha=0.85, label="MLP (TurnoverRate)", color="#55A868")
ax.bar(x + w, cnn_f1s, w, alpha=0.85, label="3D-CNN (Raw grids)", color="#DD8452")
ax.axhline(0.25, color="gray", ls="--", alpha=0.5, label="Chance (1/4)")
ax.set_xticks(x)
ax.set_xticklabels(short_names, fontsize=9)
ax.set_xlabel("Mouse ID")
ax.set_ylabel("Macro F1")
ax.set_title("Ablation across all mice: Topology vs Raw Neural Networks", fontsize=13)
ax.legend(loc="lower right", fontsize=9)
ax.set_ylim(0, 1.1)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# Summary statistics
lr_mean  = np.nanmean(lr_f1s)
mlp_mean = np.nanmean(mlp_f1s)
cnn_mean = np.nanmean(cnn_f1s)
print(f"\nMean F1 across mice:")
print(f"  LogReg (TurnoverRate): {lr_mean:.3f}")
print(f"  MLP    (TurnoverRate): {mlp_mean:.3f}")
print(f"  3D-CNN (Raw grids):    {cnn_mean:.3f}")

## 6. Summary

**Ablation design**: We compare the topological TurnoverRate representation
(720-dimensional vector summarizing topological volatility over time) against
a 3D-CNN that has access to the full raw neural activity volume
($15 \times 15 \times 10 \times T$ per trial).

| Model | Input | Parameters | What it tests |
|---|---|---|---|
| LogReg | TurnoverRate | ~3K | Linear separability of topological features |
| MLP | TurnoverRate | ~100K | Non-linear patterns in topological features |
| 1D-CNN | TurnoverRate | ~50K | Temporal structure in topological time series |
| 3D-CNN | Raw grids | ~500K | Full spatiotemporal neural activity |

**Key question**: Does the massive dimensionality reduction from
$15 \times 15 \times 10 \times 240 = 540{,}000$ raw values to a 720-dim
TurnoverRate vector lose discriminative information?

If TurnoverRate matches or beats the 3D-CNN, it demonstrates that
**topological volatility is a sufficient statistic** for stimulus 
classification in this neural population data.